# Complete MobileNetV2 training
Select your Colab GPU kernel and Run All. Upload the prepared source bundle and 224px dataset ZIP first using the official extension. This trains on the complete training split, calibrates on validation only, evaluates the held-out test split, and exports outputs. No Google Drive is used.


In [ ]:
import hashlib, json, os, sys, subprocess, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), "A CUDA GPU is required; CPU fallback is disabled."
assert Path("/content").is_dir(), "Select the existing Colab GPU kernel."
source_zip=Path("/content/colab_source.zip")
dataset_zip=Path("/content/weed_training_224.zip")
if not dataset_zip.is_file():
    transfer=json.loads(Path("/content/transfer_manifest.json").read_text())
    temporary=dataset_zip.with_suffix(".joining")
    with temporary.open("wb") as outgoing:
        for part in transfer["parts"]:
            piece=Path("/content")/part["name"]
            assert piece.is_file(), f"Missing transfer part: {piece.name}"
            content=piece.read_bytes()
            assert hashlib.sha256(content).hexdigest()==part["sha256"], f"Transfer part hash mismatch: {piece.name}"
            outgoing.write(content)
    temporary.replace(dataset_zip)
for file, expected in [(source_zip,'8ef54983a996e10695db4192ac91289f64fe6317568759ef26e3bff7270a6594')]:
    assert file.is_file(), f"Upload {file.name} with the official Colab extension."
    with file.open("rb") as stream:
        assert hashlib.file_digest(stream,"sha256").hexdigest()==expected, f"Transfer hash mismatch: {file}"
with dataset_zip.open("rb") as stream:
    image_archive_hash=hashlib.file_digest(stream,"sha256").hexdigest()
assert image_archive_hash in {'2e203e434649cb21800908f607f96b3582e8aefc892d416ba81903b5fc948c06','f89c6341456711fcc6a02ec2d41991cbce6075a0124f561bb91b969512ddc380'}, "Dataset transfer SHA256 mismatch"
project=Path("/content/AI-Weed-Assistant")
project.mkdir(exist_ok=True)
with zipfile.ZipFile(source_zip) as archive:
    for entry in archive.infolist():
        assert (project/entry.filename).resolve().is_relative_to(project.resolve())
    archive.extractall(project)
os.chdir(project)
if str(project) not in sys.path: sys.path.insert(0,str(project))
from training.prepare_runtime import prepare_runtime_dataset
data=Path("/content/weed_dataset")
final_metadata=Path("/content/final_splits.json.gz")
if image_archive_hash!='2e203e434649cb21800908f607f96b3582e8aefc892d416ba81903b5fc948c06':
    assert final_metadata.is_file(), "Upload final_splits.json.gz to apply final group-aware assignments"
    assert hashlib.sha256(final_metadata.read_bytes()).hexdigest()=='40040419ff9df9a2f19da71fcb9f48dba652791dc409ca23ca9e88bbe887c1fa', "Final split manifest hash mismatch"
    manifest=prepare_runtime_dataset(dataset_zip,data,final_metadata)
else:
    manifest=prepare_runtime_dataset(dataset_zip,data)
assert len(manifest["classes"])==12 and len(manifest["records"])==5536, "Dataset coverage mismatch"
for package in ["torchvision","numpy","scipy","sklearn","matplotlib","PIL"]:
    __import__(package)
cfg=json.loads(Path("training/config.json").read_text())
cfg.update(data_dir=str(data),output_dir="/content/weed_outputs/models/mobilenet_v2",evaluation_dir="/content/weed_outputs/evaluation/mobilenet_v2",device="cuda",num_workers=2)
resume=Path(cfg["output_dir"])/"last.pt"
uploaded_resume=Path("/content/last.pt")
if uploaded_resume.is_file() and not resume.exists():
    import shutil
    resume.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(uploaded_resume,resume)
if resume.is_file(): cfg["resume"]=str(resume)
Path("training/colab_config.json").write_text(json.dumps(cfg,indent=2))
print(json.dumps({"gpu":torch.cuda.get_device_name(0),"classes":manifest["classes"],"full_split_counts":{name:sum(r["split"]==name for r in manifest["records"]) for name in ["train","val","test"]},"config":cfg},indent=2))
subprocess.run([sys.executable,"-u","-m","training.run_job","--config","training/colab_config.json"],check=True)
